# Transformer Decoder

Decoder是Transformer架构的核心组件，用于自回归生成任务。

## 核心组件

1. **Masked Self-Attention**: 只能看到当前及之前的token（因果性）
2. **Cross-Attention**: 关注Encoder输出（可选，用于Enc-Dec架构）
3. **Feed-Forward Network**: 位置独立的全连接层
4. **Layer Normalization**: 稳定训练
5. **Residual Connection**: 改善梯度流动

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 1. 基础组件实现

In [ ]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def gelu(x):
    """GELU激活函数（GPT使用）"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))

class LayerNorm:
    def __init__(self, normalized_shape, eps=1e-5):
        self.eps = eps
        self.gamma = np.ones(normalized_shape)
        self.beta = np.zeros(normalized_shape)
    
    def forward(self, x):
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

# 测试GELU
x_test = np.linspace(-3, 3, 100)
y_gelu = gelu(x_test)
y_relu = np.maximum(0, x_test)

plt.figure(figsize=(10, 5))
plt.plot(x_test, y_gelu, label='GELU', linewidth=2)
plt.plot(x_test, y_relu, label='ReLU', linewidth=2, linestyle='--')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('GELU vs ReLU Activation Functions')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("GELU特点:")
print("  ✓ 更平滑（可微）")
print("  ✓ 负值不完全归零")
print("  ✓ GPT、BERT使用")

## 2. Masked Self-Attention（因果注意力）

In [ ]:
class MultiHeadSelfAttention:
    """带Causal Mask的多头自注意力"""
    
    def __init__(self, embed_dim, num_heads):
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # 投影矩阵
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def split_heads(self, x):
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 0, 2)
    
    def combine_heads(self, x):
        x = x.transpose(1, 0, 2)
        seq_len = x.shape[0]
        return x.reshape(seq_len, self.embed_dim)
    
    def forward(self, x, mask=None, return_weights=False):
        # 投影
        Q = np.dot(x, self.W_q.T)
        K = np.dot(x, self.W_k.T)
        V = np.dot(x, self.W_v.T)
        
        # 分割头
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # 计算attention
        outputs = []
        all_weights = []
        
        for i in range(self.num_heads):
            scores = np.dot(Q[i], K[i].T) / np.sqrt(self.head_dim)
            
            if mask is not None:
                scores = np.where(mask == 0, -1e9, scores)
            
            attn_weights = softmax(scores, axis=-1)
            output_i = np.dot(attn_weights, V[i])
            
            outputs.append(output_i)
            all_weights.append(attn_weights)
        
        # 合并
        multi_head_output = np.stack(outputs, axis=0)
        concatenated = self.combine_heads(multi_head_output)
        output = np.dot(concatenated, self.W_o.T)
        
        if return_weights:
            return output, np.stack(all_weights, axis=0)
        return output

# 测试Causal Mask
def create_causal_mask(seq_len):
    return np.tril(np.ones((seq_len, seq_len)))

seq_len = 8
causal_mask = create_causal_mask(seq_len)

# 可视化
plt.figure(figsize=(8, 6))
sns.heatmap(causal_mask, annot=True, fmt='.0f', cmap='Blues', cbar=False,
           square=True, linewidths=1)
plt.title('Causal Mask (下三角矩阵)')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.show()

print("Causal Mask特性:")
print("  ✓ 每个位置只能看到自己和之前的位置")
print("  ✓ 保证自回归生成的因果性")
print("  ✓ 下三角矩阵（1=可见，0=mask）")

## 3. Cross-Attention（交叉注意力）

In [ ]:
class MultiHeadCrossAttention:
    """多头交叉注意力"""
    
    def __init__(self, embed_dim, num_heads):
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def split_heads(self, x):
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 0, 2)
    
    def combine_heads(self, x):
        x = x.transpose(1, 0, 2)
        seq_len = x.shape[0]
        return x.reshape(seq_len, self.embed_dim)
    
    def forward(self, query, key_value, return_weights=False):
        """Q来自Decoder，K/V来自Encoder"""
        # Query来自Decoder
        Q = np.dot(query, self.W_q.T)
        # Key和Value来自Encoder
        K = np.dot(key_value, self.W_k.T)
        V = np.dot(key_value, self.W_v.T)
        
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        outputs = []
        all_weights = []
        
        for i in range(self.num_heads):
            scores = np.dot(Q[i], K[i].T) / np.sqrt(self.head_dim)
            attn_weights = softmax(scores, axis=-1)
            output_i = np.dot(attn_weights, V[i])
            
            outputs.append(output_i)
            all_weights.append(attn_weights)
        
        multi_head_output = np.stack(outputs, axis=0)
        concatenated = self.combine_heads(multi_head_output)
        output = np.dot(concatenated, self.W_o.T)
        
        if return_weights:
            return output, np.stack(all_weights, axis=0)
        return output

# 测试Cross-Attention
cross_attn = MultiHeadCrossAttention(embed_dim=64, num_heads=4)

decoder_input = np.random.randn(8, 64)  # Decoder: 8 tokens
encoder_output = np.random.randn(12, 64)  # Encoder: 12 tokens

cross_output, cross_weights = cross_attn.forward(decoder_input, encoder_output, return_weights=True)

print(f"Cross-Attention:")
print(f"  Decoder输入: {decoder_input.shape}")
print(f"  Encoder输出: {encoder_output.shape}")
print(f"  Cross-Attention输出: {cross_output.shape}")
print(f"  注意力权重: {cross_weights.shape}")

# 可视化第一个头的注意力
plt.figure(figsize=(10, 6))
sns.heatmap(cross_weights[0], cmap='YlOrRd', cbar=True)
plt.title('Cross-Attention Weights (Head 0)')
plt.xlabel('Encoder Positions (12)')
plt.ylabel('Decoder Positions (8)')
plt.show()

print("\nCross-Attention特点:")
print("  ✓ Decoder关注Encoder的输出")
print("  ✓ Q来自Decoder，K/V来自Encoder")
print("  ✓ 用于Seq2Seq任务（翻译、摘要等）")

## 4. Feed-Forward Network

In [ ]:
class FeedForwardNetwork:
    """Position-wise FFN"""
    
    def __init__(self, embed_dim, ffn_dim=None):
        if ffn_dim is None:
            ffn_dim = 4 * embed_dim
        
        self.W1 = np.random.randn(ffn_dim, embed_dim) / np.sqrt(embed_dim)
        self.b1 = np.zeros(ffn_dim)
        self.W2 = np.random.randn(embed_dim, ffn_dim) / np.sqrt(ffn_dim)
        self.b2 = np.zeros(embed_dim)
    
    def forward(self, x):
        # 第一层 + GELU
        hidden = np.dot(x, self.W1.T) + self.b1
        hidden = gelu(hidden)
        
        # 第二层
        output = np.dot(hidden, self.W2.T) + self.b2
        return output

# 测试
ffn = FeedForwardNetwork(embed_dim=512)
x_ffn = np.random.randn(10, 512)
output_ffn = ffn.forward(x_ffn)

print(f"FFN:")
print(f"  输入: {x_ffn.shape}")
print(f"  中间层: (10, {4*512})")
print(f"  输出: {output_ffn.shape}")
print(f"\n特点:")
print(f"  ✓ 对每个位置独立应用")
print(f"  ✓ 中间层通常是4×embed_dim")
print(f"  ✓ 使用GELU激活")

## 5. 完整Decoder Layer

In [ ]:
class TransformerDecoderLayer:
    """单层Transformer Decoder"""
    
    def __init__(self, embed_dim, num_heads, ffn_dim=None, has_cross_attention=False):
        self.has_cross_attention = has_cross_attention
        
        # 1. Self-Attention
        self.self_attn = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ln1 = LayerNorm(embed_dim)
        
        # 2. Cross-Attention（可选）
        if has_cross_attention:
            self.cross_attn = MultiHeadCrossAttention(embed_dim, num_heads)
            self.ln2 = LayerNorm(embed_dim)
        
        # 3. FFN
        self.ffn = FeedForwardNetwork(embed_dim, ffn_dim)
        self.ln3 = LayerNorm(embed_dim)
    
    def forward(self, x, encoder_output=None, causal_mask=None):
        # 1. Self-Attention + Residual + LN
        attn_output = self.self_attn.forward(x, mask=causal_mask)
        x = self.ln1.forward(x + attn_output)
        
        # 2. Cross-Attention + Residual + LN（如果有）
        if self.has_cross_attention and encoder_output is not None:
            cross_output = self.cross_attn.forward(x, encoder_output)
            x = self.ln2.forward(x + cross_output)
        
        # 3. FFN + Residual + LN
        ffn_output = self.ffn.forward(x)
        x = self.ln3.forward(x + ffn_output)
        
        return x

# 测试两种风格
x_test = np.random.randn(10, 512)
encoder_test = np.random.randn(15, 512)
mask_test = create_causal_mask(10)

# Decoder-only (GPT风格)
gpt_layer = TransformerDecoderLayer(512, 8, has_cross_attention=False)
gpt_output = gpt_layer.forward(x_test, causal_mask=mask_test)

# Encoder-Decoder (T5风格)
t5_layer = TransformerDecoderLayer(512, 8, has_cross_attention=True)
t5_output = t5_layer.forward(x_test, encoder_output=encoder_test, causal_mask=mask_test)

print("Decoder Layer测试:")
print(f"\nGPT风格（Decoder-only）:")
print(f"  输入: {x_test.shape}")
print(f"  输出: {gpt_output.shape}")

print(f"\nT5风格（Encoder-Decoder）:")
print(f"  Decoder输入: {x_test.shape}")
print(f"  Encoder输出: {encoder_test.shape}")
print(f"  输出: {t5_output.shape}")

## 6. 完整Transformer Decoder

In [ ]:
class TransformerDecoder:
    """完整Transformer Decoder"""
    
    def __init__(self, num_layers, embed_dim, num_heads, ffn_dim=None,
                 has_cross_attention=False, vocab_size=None):
        self.num_layers = num_layers
        
        # 堆叠多层
        self.layers = [
            TransformerDecoderLayer(embed_dim, num_heads, ffn_dim, has_cross_attention)
            for _ in range(num_layers)
        ]
        
        # 最终LN
        self.final_ln = LayerNorm(embed_dim)
        
        # LM Head
        if vocab_size is not None:
            self.lm_head = np.random.randn(vocab_size, embed_dim) / np.sqrt(embed_dim)
        else:
            self.lm_head = None
    
    def forward(self, x, encoder_output=None, causal_mask=None, return_logits=False):
        # 逐层处理
        for layer in self.layers:
            x = layer.forward(x, encoder_output, causal_mask)
        
        # 最终LN
        x = self.final_ln.forward(x)
        
        # Logits
        if return_logits and self.lm_head is not None:
            logits = np.dot(x, self.lm_head.T)
            return logits
        
        return x

# 创建GPT风格的Decoder
gpt_decoder = TransformerDecoder(
    num_layers=12,
    embed_dim=768,
    num_heads=12,
    has_cross_attention=False,
    vocab_size=50000
)

x_input = np.random.randn(10, 768)
mask_input = create_causal_mask(10)

hidden_output = gpt_decoder.forward(x_input, causal_mask=mask_input, return_logits=False)
logits_output = gpt_decoder.forward(x_input, causal_mask=mask_input, return_logits=True)

print("完整GPT风格Decoder:")
print(f"  配置: 12层, 768维, 12头")
print(f"  输入: {x_input.shape}")
print(f"  Hidden输出: {hidden_output.shape}")
print(f"  Logits输出: {logits_output.shape}")

## 7. 参数量分析

In [ ]:
def count_decoder_params(embed_dim, num_heads, num_layers, vocab_size, has_cross_attn):
    """计算Decoder参数量"""
    # 每层参数
    params_per_layer = 0
    
    # Self-Attention: 4 * d²
    params_per_layer += 4 * embed_dim * embed_dim
    
    # Cross-Attention: 4 * d²（如果有）
    if has_cross_attn:
        params_per_layer += 4 * embed_dim * embed_dim
    
    # FFN: 2 * d * 4d
    params_per_layer += 2 * embed_dim * (4 * embed_dim)
    
    # LayerNorm: 2d per LN
    ln_per_layer = 3 if has_cross_attn else 2
    params_per_layer += ln_per_layer * 2 * embed_dim
    
    # 总参数
    total = num_layers * params_per_layer
    total += 2 * embed_dim  # 最终LN
    total += vocab_size * embed_dim  # LM Head
    
    return total, params_per_layer

# 不同模型配置
models = [
    ('GPT-2 Small', 12, 768, 12, 50257, False),
    ('GPT-2 Medium', 24, 1024, 16, 50257, False),
    ('GPT-2 Large', 36, 1280, 20, 50257, False),
    ('GPT-2 XL', 48, 1600, 25, 50257, False),
    ('GPT-3', 96, 12288, 96, 50257, False),
]

results = []
for name, layers, dim, heads, vocab, cross in models:
    total, per_layer = count_decoder_params(dim, heads, layers, vocab, cross)
    results.append((name, total))

# 可视化
names, params = zip(*results)
params_b = [p / 1e9 for p in params]

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(names)), params_b, color='steelblue')
plt.xticks(range(len(names)), names, rotation=45, ha='right')
plt.ylabel('Parameters (Billions)')
plt.title('Transformer Decoder Parameter Count by Model')
plt.grid(axis='y', alpha=0.3)
plt.yscale('log')

# 添加数值标签
for bar, val in zip(bars, params_b):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}B',
            ha='center', va='bottom')

plt.tight_layout()
plt.show()

# 打印表格
print("\n模型参数量对比:")
print(f"{'模型':<15} {'参数量':<15}")
print("-" * 30)
for name, p in results:
    print(f"{name:<15} {p/1e9:>10.1f}B")

## 8. Decoder架构对比

In [ ]:
# 创建对比图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Decoder-only (GPT)
gpt_blocks = ['Input', 'Self-Attn\n(Masked)', 'Add & Norm', 'FFN', 'Add & Norm', 'Output']
gpt_y = list(range(len(gpt_blocks)))

ax1.barh(gpt_y, [1]*len(gpt_blocks), color='lightblue')
for i, block in enumerate(gpt_blocks):
    ax1.text(0.5, i, block, ha='center', va='center', fontsize=10, weight='bold')
ax1.set_yticks([])
ax1.set_xticks([])
ax1.set_title('Decoder-only (GPT)', fontsize=12, weight='bold')
ax1.set_xlim(0, 1)

# Encoder-Decoder (T5)
t5_blocks = ['Input', 'Self-Attn\n(Masked)', 'Add & Norm', 'Cross-Attn', 'Add & Norm', 'FFN', 'Add & Norm', 'Output']
t5_y = list(range(len(t5_blocks)))

ax2.barh(t5_y, [1]*len(t5_blocks), color='lightgreen')
for i, block in enumerate(t5_blocks):
    ax2.text(0.5, i, block, ha='center', va='center', fontsize=10, weight='bold')
ax2.set_yticks([])
ax2.set_xticks([])
ax2.set_title('Encoder-Decoder (T5)', fontsize=12, weight='bold')
ax2.set_xlim(0, 1)

plt.tight_layout()
plt.show()

print("架构对比:")
print("\nDecoder-only (GPT):")
print("  ✓ 只有Masked Self-Attention")
print("  ✓ 用于自回归生成")
print("  ✓ 应用: GPT、LLaMA、PaLM")

print("\nEncoder-Decoder (T5):")
print("  ✓ 包含Cross-Attention")
print("  ✓ 用于Seq2Seq任务")
print("  ✓ 应用: T5、BART、mT5")

## 9. 总结

### Transformer Decoder关键特性

1. **Masked Self-Attention**: 保证因果性（自回归）
2. **Cross-Attention**: 连接Encoder和Decoder（可选）
3. **Position-wise FFN**: 对每个位置独立处理
4. **Residual + LayerNorm**: 稳定训练和梯度流动

### 两种架构

- **Decoder-only**: GPT、LLaMA、PaLM等（主流LLM）
- **Encoder-Decoder**: T5、BART（Seq2Seq任务）

### 实际应用

- 文本生成、对话、代码生成
- 机器翻译、摘要、问答
- 所有自回归语言模型的核心